In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

In [ ]:
# 텍스트 파일 읽기
with open('./dataset/history.txt', 'r', encoding='utf-8') as file :
    text = file.read()

In [ ]:
# 워드 클라우드 설정 및 생성
wordcloud = WordCloud(
    font_path='malgun', # 한글 폰트 설정 (맑은 고딕)
    background_color = 'white',
    width = 800,
    height= 600,
    max_words= 200,
    max_font_size=100,
    min_font_size=10,
    random_state=42
).generate(text)

In [ ]:
# 워드 클라우드 이미지 시각화
plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
import gradio as gr
from wordcloud import WordCloud
import os

In [ ]:
# 워드 클라우드 생성 함수
def generate_wordcloud(file_obj) :
    try :
        # 파일이 없는 경우 처리
        if file_obj is None :
            return None
        
        # Gradio의 파일 객체에서 파일 경로 가져오기
        file_path = file_obj.name

        # 파일읽기
        with open(file_path, 'r', encoding='utf-8') as file :
            text = file.read()
        
        # 폰트 경로설정
        # 'malgun' 이름만 쓰는 대신 시스템의 실제 폰트 파일 경로를 지정해야 안정적입니다.
        # 1. 윈도우 (맑은 고딕)
        font_path = "C:/Windows/Fonts/malgun.ttf"

        # 2. 윈도우에 폰트가 없으면 다른 OS 경로 탐색 (예 : macOs, Linux)
        if not os.path.exists(font_path) :
            # macOs (Apple SD Gothic Neo)
            font_path="/System/Library/Fonts/AppleSDGothicNeo.ttc"
        if not os.path.exists(font_path):
            # Linux (NanumGothic - 'fonts-nanum' 패키지 설치 필요)
            font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
        if not os.path.exists(font_path):
            # 폰트를 못찾으면 None (한글이 깨짐)
            font_path = None
            print("경고: 한글 폰트(맑은 고딕 등)를 찾을 수 없습니다. 한글이 네모로 표시될 수 있습니다.")
        
        # 워드 클라우드 생성
        wordcloud = WordCloud(
            font_path=font_path, # 위에서 찾은 실제 폰트 경로 사용
            background_color='white',
            width=800,
            height=600,
            max_words=200,
            max_font_size=100,
            min_font_size=10,
            random_state=42
        ).generate(text)

        return wordcloud.to_image()
    except Exception as e :
        print(f'ERROR : {str(e)}')
        return None


In [ ]:
# Gradio 인터페이스 생성
iface = gr.Interface(
    fn=generate_wordcloud,
    inputs=gr.File(label='Upload a .txt file'),
    outputs=gr.Image(type='pil', label="Word Cloud")
)

In [ ]:
iface.launch(server_port=7861, share=True, server_name="0.0.0.0")

In [ ]:
iface.close()

In [ ]:
import gradio as gr
import ollama

In [ ]:
def analyze_image(image_path, prompt) :
    """ 
    업로드된 이미지와 프롬프트를 Ollama 모델에 전달하여 결과를 반환하는 함수
    """
    if not image_path:
        return "이미지를 먼저 업로드해주세요"
    try :
        # Ollama API 를 호출하여 이미지와 텍스트를 함께 전송
        response = ollama.chat(
            model = 'gemma4:e2b',
            messages = [
                {
                    'role' : 'user',
                    'content' : prompt,
                    'images' : [image_path] # Gradio에서 전달 받은 이미지 파일 경로
                }
            ]
        )
        return response['message']['content']
    
    except Exception as e :
        error_msg = f'오류가 발생했습니다 : {str(e)}\n\n'
        error_msg += "💡 참고: 입력하신 'gemma4:e2b' 모델이 이미지 인식(멀티모달)을 지원하는 비전 모델인지 확인해 주세요. "
        error_msg += "(예: 일반적인 Gemma 텍스트 모델은 이미지를 처리하지 못하며, PaliGemma 같은 모델이 필요합니다.)"
        return error_msg

In [ ]:
# Gradio 인터페이스 구성
with gr.Blocks(theme=gr.themes.Soft()) as demo :
    gr.Markdown("# Gemma 이미지 인식기 (gemma4:e2b)")
    gr.Markdown('이미지를 업로드하고 질문을 입력하면 모델이 이미지를 분석하여 답변합니다.')

    with gr.Row() :
        with gr.Column():
            # 이미지 입력 컴포넌트 (파일 경로 형태로 전달되도록 설정)
            input_image = gr.Image(type='filepath', label='이미지 업로드')
            input_prompt = gr.Textbox(
                label="질문 (Prompt)",
                value='이 이미지에 대해 자세히 설명해줘',
                placeholder='이미지에 대해 묻고 싶은 것을 입력하세요'
            )
            submit_btn = gr.Button('분석하기',variant='primary')

        with gr.Column() :
            # 결과 출력 컴포넌트
            output_text = gr.Textbox(label="분석 결과", lines=10)

        # 버튼 클릭 시 동작 설정
        submit_btn.click(
            fn=analyze_image,
            inputs=[input_image, input_prompt],
            outputs=output_text
        )

In [ ]:
demo.launch()

In [ ]:
demo.close()

In [ ]:
import ollama

In [ ]:
def ask_gemma(question) :
    # ollama를 사용하여 모델로부터 응답 생성
    chatbot_role = '너는 전문 심리 상담가야. 질문에 대한 답은 3줄 이내로 짧게 해줘'
    response = ollama.chat(model="gemma4:e2b", messages=[
        {"role" : 'system', 'content' : chatbot_role}, # 챗봇의 기본 역할 부여
        {'role' : 'user', 'content' : question}, # 질문
    ])
    return response['message']['content']

In [ ]:
question = '행복하기 위해 어떻게 하면 좋을까?'
response = ask_gemma(question)
print(response)

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage   # HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
import gradio as gr

In [ ]:
# ChatOllama 모델 초기화
model = ChatOllama(model="gemma4:e2b", temperature=0.7, verbose=False)
# temperture가 낮을 수록 거의 동일답변, 높을 수록 창의적인 답변

In [ ]:
# 채팅 기록을 포함하여 응답을 생성하는 함수
def chat(message, history) :
    # 이전 대화 기록을 chat이 Ollama 형식으로 변환
    chat_history = []
    for human, ai in history :
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 현재 메시지 추가
    chat_history.append(HumanMessage(content=message))

    # 모델을 사용하여 응답 생성
    response = model.invoke(chat_history)

    return response.content

In [ ]:
# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        '안녕하세요!',
        '인공지능에 대해 설명해주세요',
        '파이썬의 장점은 무엇인가요?'
    ], 
    title = "AI 챗봇",
    description= "질문을 입력하면 AI가 답변합니다."
)

In [ ]:
# 서버 실행
demo.launch(server_port=7777, server_name="0.0.0.0")

In [ ]:
demo.close()

In [ ]:
import pandas as pd
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage  # HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter   # 특정 문자(예: 줄바꿈, 공백)를 기준으로 텍스트를 분할하는 기능을 제공
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
import gradio as gr

In [ ]:
df = pd.read_csv("./dataset/indata_kor2.csv", encoding='utf-8')

In [ ]:
df.tail()

In [ ]:
# 텍스트 분할
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)  # 텍스트 조각 최대 1000자, 텍스트 조각 사이에 200자만큼의 중복을 허용(문맥 유지)
texts = text_splitter.split_text("\n".join(df.to_string()))   # 문자열들을 줄바꿈 문자(\n)를 기준으로 연결

In [ ]:
# 임베딩 모델 초기화
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")
# 모델 이름 : 조직이름(sentence-transformers) 다양한 작업 가능(all)-MS사 경령화 트랜스포머모델(MiniLM)-모델의 레이어수(L6)-모델이 버전(v2)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
vectorstore = FAISS.from_texts(texts, embeddings)  # from_texts : 임베딩으로 변환된 벡터를 FAISS 인덱스에 저장

In [ ]:
# ChatOllama 모델 초기화
llm = ChatOllama(model="gemma2", tempeature=0.1)  # temperture가 낮을수록 거의 동일답변, 높을수록 창의적인 답변

In [ ]:
# 프롬프트 템플릿 정의 : 모델이 제공된 Context(청크 검색 결과) 만을 기반으로 답변하도록 유도
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer based on the provided context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}\n\nContext: {context}")
])

In [ ]:
# 문서 포맷팅 : RAG 시스템에서 검색된 문서들(docs)을 LLM이 이해하기 좋은 형태의 하나의 문자열로 변환
def format_docs(docs):
    if not docs:
        return "No context available"
    result = [] # 결과를 모아둘 빈 리스트 생성
    for doc in docs:
        if hasattr(doc, 'page_content'):  # 객체(doc)가 특정 속성(page_content)을 가지고 있는지 확인
            result.append(doc.page_content)
    return "\n".join(result)

In [ ]:
# 리트리버 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
def chat(message, history):
    # 이전 대화 기록을 메시지 형식으로 변환
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 문서 검색
    docs = retriever.invoke(message)
    context = format_docs(docs)

    # 프롬프트 생성
    messages = prompt.format_messages(
        chat_history=chat_history,
        question=message,
        context=context
    )

    # 모델 응답 생성
    response = llm.invoke(messages)

    # 소스 문서 정보 추출
    sources = set([doc.metadata.get('source', 'Unknown') for doc in docs])
    source_info = f"\n\n참고 출처: {', '.join(sources)}" if sources else ""

    return response.content + source_info

In [ ]:
# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?",
        "스마트금융과에 대해 설명해주세요",
        "한국폴리텍대한 추천할만한 학과 하나를 소개해주세요."
    ],
    title="대학 정보 AI 챗봇",
    description="스마트금융과에 대한 질문을 입력하면 AI가 CSV데이터를 참고하여 한글로 답변합니다."
)

In [ ]:
# 서버 실행
demo.launch(server_port=7861, server_name="0.0.0.0")

In [ ]:
demo.close()

In [ ]:
import os
from dotenv import load_dotenv  # 환경변수 로드가 필요한 경우
import whisper
import gradio as gr

In [ ]:
os.environ['PATH'] += os.pathsep + "D:\python_sim\ffmpeg\bin\ffmpeg.exe"
os.environ["FFMPEG_BINARY"] = r"C:\python_sim\ffmpeg\bin\ffmpeg.exe"

In [ ]:
def transcribe_audio(audio_path) :
    # Whisper 모델 로드
    model = whisper.load_model('base') # tiny, base(Vram:~1GB).small(Varam:~2GB). medium(Vram:~5G), Large(Vram:~10GB)

    # 오디오 파일 전사
    result = model.transcribe(audio_path)

    # 전사된 텍스트 반환
    return result["text"]

In [ ]:
def process_audio(audio) :
    if audio is None :
        return "오디오 파일을 업로드해주세요"
    try :
        transcribed_text = transcribe_audio(audio)
        return transcribed_text
    except Exception as e :
        return f'오류가 발생했습니다 : {str(e)}'

In [ ]:
# Gradio 인터페이스 생성
iface = gr.Interface(
    fn=process_audio,
    inputs=gr.Audio(type='filepath', label='MP3 파일 업로드'),
    outputs=gr.Textbox(label="음성 분석 결과", lines=10),
    title= "MP3 to Text Converter",
    description = "MP3 파일을 업로드하면 텍스트로 변환합니다"
)

In [7]:
# 디버그 모드로 Gradio 인터페이스 실행
iface.launch(server_port=7861, server_name="0.0.0.0", debug=True)

 51%|███████████████████▌                  | 71.3M/139M [01:00<00:38, 1.84MiB/s]

Keyboard interruption in main thread... closing server.


 52%|███████████████████▌                  | 71.5M/139M [01:00<00:36, 1.92MiB/s]

 54%|████████████████████▌                 | 74.9M/139M [01:03<00:43, 1.54MiB/s]

In [ ]:
iface.close()

Closing server running on port: 7861


 83%|█████████████████████████████████       | 115M/139M [01:34<00:29, 850kiB/s]

In [ ]:
# 사전 설치 : pip install gtts
import gradio as gr
from gtts import gTTS
import os
import tempfile

 83%|█████████████████████████████████▏      | 115M/139M [01:34<00:39, 629kiB/s]

 84%|█████████████████████████████████▍      | 116M/139M [01:36<00:33, 710kiB/s]

In [12]:
def text_to_speech(text, lang='ko'):
    # 임시 파일 생성
    with tempfile.NamedTemporaryFile(delete=False, suffix='.mp3') as fp:
        temp_filename = fp.name

    # TTS 변환
    tts = gTTS(text=text, lang=lang)
    tts.save(temp_filename)
    return temp_filename

In [13]:
def process_tts(text, lang):
    if not text:
        return None, "텍스트를 입력해주세요."
    try:
        audio_file = text_to_speech(text, lang)
        return audio_file, "변환이 완료되었습니다. 아래에서 재생 또는 다운로드할 수 있습니다."
    except Exception as e:
        return None, f"오류가 발생했습니다: {str(e)}"

In [14]:
# Gradio 인터페이스 생성
iface = gr.Interface(
    fn=process_tts,
    inputs=[
        gr.Textbox(lines=5, label="텍스트 입력"),
        gr.Dropdown(choices=['ko', 'en', 'ja', 'zh-cn'], label="언어 선택", value='ko')
    ],
    outputs=[
        gr.Audio(label="생성된 오디오"),
        gr.Textbox(label="상태 메시지")
    ],
    title = "Text to Speech Converter",
    description="텍스트를 입력하연 MP3 파일로 변환합니다."
)

In [ ]:
# 디버그 모드로 Gradio 인터페이스 실행
iface.launch(server_port=7861, server_name="0.0.0.0",share=True, debug=True)

* Running on local URL:  http://0.0.0.0:7861
* Running on public URL: https://e9583bd0ef8037df32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
iface.close()